In [1]:
import os
from transformers import AutoConfig, AutoTokenizer, AutoModelForSequenceClassification
os.environ['CUDA_VISIBLE_DEVICES'] = '2,3,4,5,6,7'
tokenizer = AutoTokenizer.from_pretrained(
    "/pf9550-bdp-A800/zhengyulong/HyenaModel/HyenaBase", trust_remote_code=True
)

In [3]:
tokenizer(["ACTG","ACGTGTCA"],padding="max_length",max_length=10)

{'input_ids': [[4, 4, 4, 4, 4, 7, 8, 10, 9, 1], [4, 7, 8, 9, 10, 9, 10, 8, 7, 1]]}

In [2]:
!bash

(base) ]0;zhengyulong@bdp-gpu04: /pf9550-bdp-A800/zhengyulong/lyrazhengyulong@bdp-gpu04:/pf9550-bdp-A800/zhengyulong/lyra$ ^C
(base) ]0;zhengyulong@bdp-gpu04: /pf9550-bdp-A800/zhengyulong/lyrazhengyulong@bdp-gpu04:/pf9550-bdp-A800/zhengyulong/lyra$ ^C

(base) ]0;zhengyulong@bdp-gpu04: /pf9550-bdp-A800/zhengyulong/lyrazhengyulong@bdp-gpu04:/pf9550-bdp-A800/zhengyulong/lyra$ 

In [1]:
import os
# 重要：确保这是在 import s4d_cuda_kernel 之前的第一个代码单元格执行
os.environ['LD_LIBRARY_PATH'] = '/home/zhengyulong/miniconda3/lib/python3.9/site-packages/torch/lib:' + os.environ.get('LD_LIBRARY_PATH', '')

# 然后再尝试导入
import s4d_cuda_kernel
print("Import successful!")
print(dir(s4d_cuda_kernel))

ImportError: libc10.so: cannot open shared object file: No such file or directory

In [1]:
import torch
import s4d_cuda_kernel
print("Import successful!")
print(dir(s4d_cuda_kernel))

Import successful!
['__doc__', '__file__', '__loader__', '__name__', '__package__', '__spec__', 'compute_K_autograd', 'fft_conv1d_forward']


In [ ]:
import s4d_cuda_kernel # Import at module level or within class __init__

# Inside S4DKernel.forward(self, L):
    # ... (calculate dt, A etc. if needed elsewhere, or remove if only for K)

    # Replace the PyTorch K calculation:
    # dt = torch.exp(self.log_dt)
    # C = torch.view_as_complex(self.C) # Keep self.C as (H, N2, 2)
    # A = -torch.exp(self.log_A_real) + 1j * self.A_imag
    # dtA = A * dt.unsqueeze(-1)
    # K_pytorch = dtA.unsqueeze(-1) * torch.arange(L, device=A.device) # OLD
    # C_pytorch = C * (torch.exp(dtA) - 1.) / A # OLD
    # K_pytorch = 2 * torch.einsum('hn, hnl -> hl', C_pytorch, torch.exp(K_pytorch)).real # OLD

    # Use CUDA Autograd version:
    # Ensure inputs are on the correct device (CUDA)
    log_dt_cuda = self.log_dt.cuda()
    C_cuda = self.C.cuda() # self.C is already (H, N2, 2)
    log_A_real_cuda = self.log_A_real.cuda()
    A_imag_cuda = self.A_imag.cuda()

    # Make sure tensors passed to CUDA extension are contiguous
    log_dt_cuda = log_dt_cuda.contiguous()
    C_cuda = C_cuda.contiguous()
    log_A_real_cuda = log_A_real_cuda.contiguous()
    A_imag_cuda = A_imag_cuda.contiguous()


    K = s4d_cuda_kernel.compute_K_autograd(
        log_dt_cuda,
        C_cuda, # Pass the real tensor directly
        log_A_real_cuda,
        A_imag_cuda,
        L
    )
    # K is now computed using CUDA with autograd support

    # The rest of the S4D forward (FFT conv, skip connection, etc.) can remain the same
    # using torch.fft which supports autograd.
    # ...
    k_f = torch.fft.rfft(K.float(), n=2*L) # Ensure K is float32 if needed
    u_f = torch.fft.rfft(u.float(), n=2*L) # Ensure u is float32
    y = torch.fft.irfft(u_f * k_f, n=2*L)[..., :L]
    # ... rest of S4D forward ...

    return y